In [0]:
from pyspark.sql import functions as F

# 1. Leer Silver
df = spark.table(
    "open_meteo.api_open.silver_weather_hourly"
)

# 2. Calcular indicadores diarios
gold = (
    df
    .withColumn("weather_date", F.to_date("weather_time"))
    .groupBy("weather_date")
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temperature_c"),
        F.min("temperature_c").alias("min_temperature_c"),
        F.max("temperature_c").alias("max_temperature_c"),
        F.round(F.avg("humidity_pct"), 2).alias("avg_humidity_pct"),
        F.round(F.sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        F.round(F.avg("wind_speed_kmh"), 2).alias("avg_wind_speed_kmh")
    )
)

# 3. Guardar Gold
gold.write.format("delta").mode("overwrite").saveAsTable(
    "open_meteo.api_open.gold_weather_daily"
)

display(gold.orderBy("weather_date"))